### CNN Feature Extraction | EEG only

In [ ]:
# =========================================================
# EEG-ONLY FULL PIPELINE 🚀 
# =========================================================

from pathlib import Path
from collections import OrderedDict
import gc, math, random
import numpy as np

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, Sampler

from sklearn.metrics import f1_score, balanced_accuracy_score
from tqdm import tqdm

# =========================================================
# 0) Setup
# =========================================================
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(torch.cuda.current_device())
    print("GPU:", props.name)
    print("VRAM (GB):", round(props.total_memory / (1024**3), 2))
print("Torch:", torch.__version__)

# =========================================================
# 1) Paths
# =========================================================
SPLIT_ROOT = Path(r"C:\SeniorProject\Data\NPZ_SPLITS_FILTERED_CAP1H_IMPD3")

TRAIN_ROOT = SPLIT_ROOT / "TRAIN" / "EEG"
VAL_ROOT   = SPLIT_ROOT / "VAL"   / "EEG"
TEST_ROOT  = SPLIT_ROOT / "TEST"  / "EEG"

PACKED_ROOT = SPLIT_ROOT / "EEG_CNN_PACKED"
FEAT_ROOT   = SPLIT_ROOT / "EEG_CNN_FEATURES"
CKPT_DIR    = SPLIT_ROOT / "EEG_CNN_CKPTS"

for s in ["TRAIN", "VAL", "TEST"]:
    (PACKED_ROOT / s).mkdir(parents=True, exist_ok=True)
    (FEAT_ROOT / s).mkdir(parents=True, exist_ok=True)

CKPT_DIR.mkdir(parents=True, exist_ok=True)
BEST_CKPT_PATH = CKPT_DIR / "best.pt"

# =========================================================
# 2) Config
# =========================================================
NUM_CLASSES = 3
BATCH_SIZE = 32
MAX_EPOCHS = 80
MIN_EPOCHS = 15
PATIENCE = 10
LR = 5e-4
WEIGHT_DECAY = 1e-4

RUN_CACHE_SIZE = 8
FEAT_BATCH_SIZE = 64

# =========================================================
# 3) PACKING
# =========================================================
def pack_split(src_root, dst_root):
    files = sorted(src_root.glob("*.npz"))

    saved, skipped = 0, 0

    for f in tqdm(files, desc=f"Packing {src_root.name}"):
        out = dst_root / f.name

        if out.exists():
            skipped += 1
            continue

        npz = np.load(f, allow_pickle=True)

        X = np.nan_to_num(np.asarray(npz["X"], dtype=np.float32), nan=0.0, posinf=0.0, neginf=0.0)
        y = np.asarray(npz["y"], dtype=np.int8)

        if "win_idx" in npz:
            win_idx = np.asarray(npz["win_idx"], dtype=np.int32)
        else:
            win_idx = np.arange(len(y), dtype=np.int32)

        np.savez_compressed(
            out,
            X_EEG=X,
            y=y,
            win_idx=win_idx,
            base=np.array([f.stem], dtype=object)
        )

        saved += 1

    print(f"✅ Packed {src_root.name}: saved={saved}, skipped={skipped}")

# =========================================================
# 4) Dataset
# =========================================================
class PackedEEGDataset(Dataset):
    def __init__(self, root, cache_size=RUN_CACHE_SIZE):
        self.files = sorted(Path(root).glob("*.npz"))
        if len(self.files) == 0:
            raise RuntimeError(f"No packed files found in {root}")

        self.cache = OrderedDict()
        self.cache_size = int(cache_size)

        self.flat = []
        self.run_to_indices = []

        for i, f in enumerate(self.files):
            try:
                npz = np.load(f, allow_pickle=True)
                n = len(npz["y"])
            except Exception as e:
                print(f"⚠️ Skip file {f.name}: {e}")
                continue

            idxs = list(range(len(self.flat), len(self.flat) + n))
            self.run_to_indices.append(idxs)

            for j in range(n):
                self.flat.append((i, j))

        if len(self.flat) == 0:
            raise RuntimeError(f"No usable samples in {root}")

    def __len__(self):
        return len(self.flat)

    def _load_run(self, i):
        if i in self.cache:
            self.cache.move_to_end(i)
            return self.cache[i]

        npz = np.load(self.files[i], allow_pickle=True)

        run = {
            "X": np.asarray(npz["X_EEG"], dtype=np.float32),
            "y": np.asarray(npz["y"], dtype=np.int8),
            "win_idx": np.asarray(npz["win_idx"], dtype=np.int32),
            "base": str(npz["base"][0]) if "base" in npz else self.files[i].stem,
        }

        self.cache[i] = run

        while len(self.cache) > self.cache_size:
            self.cache.popitem(last=False)

        return run

    def __getitem__(self, idx):
        i, j = self.flat[idx]
        run = self._load_run(i)

        X = torch.from_numpy(run["X"][j])
        y = torch.tensor(int(run["y"][j]), dtype=torch.long)
        base = run["base"]
        win_idx = torch.tensor(int(run["win_idx"][j]), dtype=torch.long)

        return {"EEG": X}, y, base, win_idx

# =========================================================
# 5) Sampler
# =========================================================
class RunSampler(Sampler):
    def __init__(self, dataset, batch_size, shuffle=True, drop_last=False):
        self.ds = dataset
        self.bs = int(batch_size)
        self.shuffle = bool(shuffle)
        self.drop_last = bool(drop_last)

    def __iter__(self):
        rng = np.random.default_rng()
        run_ids = list(range(len(self.ds.run_to_indices)))

        if self.shuffle:
            rng.shuffle(run_ids)

        batches = []

        for rid in run_ids:
            idxs = self.ds.run_to_indices[rid].copy()

            if self.shuffle:
                rng.shuffle(idxs)

            for i in range(0, len(idxs), self.bs):
                batch = idxs[i:i+self.bs]
                if len(batch) < self.bs and self.drop_last:
                    continue
                batches.append(batch)

        if self.shuffle:
            rng.shuffle(batches)

        for b in batches:
            yield b

    def __len__(self):
        total = 0
        for idxs in self.ds.run_to_indices:
            if self.drop_last:
                total += len(idxs) // self.bs
            else:
                total += math.ceil(len(idxs) / self.bs)
        return total

# =========================================================
# 6) Collate
# =========================================================
def collate_fn(batch):
    Xs, ys, bases, win_idxs = [], [], [], []

    for Xd, y, base, wi in batch:
        Xs.append(Xd["EEG"])
        ys.append(y)
        bases.append(base)
        win_idxs.append(wi)

    return {"EEG": torch.stack(Xs)}, torch.stack(ys), bases, torch.stack(win_idxs)

# =========================================================
# 7) Model
# =========================================================
class EEGModel(nn.Module):
    """
    Final extracted features for BiLSTM = 256
    Saved shape per run: F -> (n_wins, 256)
    """
    def __init__(self, in_ch, emb_dim=128, feat_dim=256, num_classes=3):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Conv1d(in_ch, 64, kernel_size=7, padding=3, bias=False),
            nn.BatchNorm1d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool1d(2),

            nn.Conv1d(64, 128, kernel_size=5, padding=2, bias=False),
            nn.BatchNorm1d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool1d(2),

            nn.Conv1d(128, 256, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True),
        )

        # First projection
        self.proj = nn.Linear(256, emb_dim)

        # Final feature block -> 256
        self.fuse = nn.Sequential(
            nn.LayerNorm(emb_dim),
            nn.Dropout(0.2),
            nn.Linear(emb_dim, feat_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),
        )

        self.classifier = nn.Linear(feat_dim, num_classes)

    def forward(self, Xd, return_feat=False):
        x = Xd["EEG"]                  # (B,C,T)
        h = self.encoder(x)            # (B,256,T')
        h = h.mean(dim=-1)             # (B,256)

        z = self.proj(h)               # (B,128)
        feat = self.fuse(z)            # (B,256)
        logits = self.classifier(feat) # (B,3)

        return logits if not return_feat else (logits, feat)

# =========================================================
# 8) Utils
# =========================================================
def compute_weights(root):
    counts = np.zeros(NUM_CLASSES, dtype=np.float64)

    for f in Path(root).glob("*.npz"):
        y = np.load(f, allow_pickle=True)["y"]
        y = np.asarray(y).astype(int)

        for c in range(NUM_CLASSES):
            counts[c] += np.sum(y == c)

    counts = np.maximum(counts, 1)
    w = counts.sum() / (NUM_CLASSES * counts)
    w = w / w.mean()

    return torch.tensor(w, dtype=torch.float32).to(DEVICE)

# =========================================================
# 9) Eval
# =========================================================
@torch.no_grad()
def evaluate(model, loader, loss_fn):
    model.eval()

    total_loss = 0.0
    total_n = 0
    all_y, all_p = [], []

    for Xd, y, _, _ in tqdm(loader, desc="Validation", leave=False):
        Xd = {k: v.to(DEVICE) for k, v in Xd.items()}
        y = y.to(DEVICE)

        logits = model(Xd)
        loss = loss_fn(logits, y)

        total_loss += float(loss.item()) * y.size(0)
        total_n += y.size(0)

        pred = torch.argmax(logits, dim=1).cpu().numpy()
        all_y.append(y.cpu().numpy())
        all_p.append(pred)

    y_true = np.concatenate(all_y)
    y_pred = np.concatenate(all_p)

    f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
    bal = balanced_accuracy_score(y_true, y_pred)
    avg_loss = total_loss / max(1, total_n)

    return avg_loss, f1, bal

# =========================================================
# 10) Train
# =========================================================
def train(model, train_loader, val_loader):
    weights = compute_weights(PACKED_ROOT / "TRAIN")
    print("Class weights:", weights.detach().cpu().numpy().tolist())

    loss_fn = nn.CrossEntropyLoss(weight=weights)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    best = -1.0
    bad = 0
    best_epoch = -1

    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        total_loss = 0.0
        total_n = 0

        for Xd, y, _, _ in tqdm(train_loader, desc=f"Train Epoch {epoch}"):
            Xd = {k: v.to(DEVICE) for k, v in Xd.items()}
            y = y.to(DEVICE)

            opt.zero_grad()
            logits = model(Xd)
            loss = loss_fn(logits, y)
            loss.backward()
            opt.step()

            total_loss += float(loss.item()) * y.size(0)
            total_n += y.size(0)

        train_loss = total_loss / max(1, total_n)
        val_loss, f1, bal = evaluate(model, val_loader, loss_fn)

        print(
            f"Epoch {epoch:03d} | "
            f"train_loss {train_loss:.4f} | "
            f"val_loss {val_loss:.4f} | "
            f"bal_acc {bal:.4f} | "
            f"f1 {f1:.4f}"
        )

        if bal > best + 1e-6:
            best = bal
            best_epoch = epoch
            bad = 0
            torch.save(model.state_dict(), BEST_CKPT_PATH)
            print(f"✅ Saved BEST (epoch={epoch}, val_bal_acc={bal:.4f})")
        else:
            bad += 1

        if epoch >= MIN_EPOCHS and bad >= PATIENCE:
            print(f"🛑 Early stopping at epoch {epoch}. Best epoch={best_epoch}, best val_bal_acc={best:.4f}")
            break

        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

# =========================================================
# 11) Feature Extraction
# =========================================================
@torch.no_grad()
def extract(model, root, out):
    """
    Saves:
      F       -> (n_wins, 256)
      y       -> (n_wins,)
      win_idx -> (n_wins,)
      base    -> base run name
    """
    model.eval()
    files = sorted(list(Path(root).glob("*.npz")))

    for f in tqdm(files, desc=f"Extract {Path(root).name}"):
        npz = np.load(f, allow_pickle=True)

        X = np.asarray(npz["X_EEG"], dtype=np.float32)
        y = np.asarray(npz["y"], dtype=np.int8)
        win_idx = np.asarray(npz["win_idx"], dtype=np.int32)
        base = str(npz["base"][0]) if "base" in npz else f.stem

        feats = []

        for i in tqdm(range(0, len(X), FEAT_BATCH_SIZE), leave=False, desc="Batches"):
            batch = {
                "EEG": torch.from_numpy(X[i:i+FEAT_BATCH_SIZE]).to(DEVICE)
            }

            _, feat = model(batch, return_feat=True)
            feats.append(feat.detach().cpu().numpy().astype(np.float32))

        F = np.concatenate(feats, axis=0).astype(np.float32)

        print(f"{f.name} -> F shape = {F.shape}")   # expected: (n_wins, 256)

        np.savez_compressed(
            out / f.name,
            F=F,
            y=y,
            win_idx=win_idx,
            base=np.array([base], dtype=object)
        )

    print(f"✅ Features saved -> {out}")

# =========================================================
# 12) Main
# =========================================================
def main():
    print("\n[1/4] Packing...")
    pack_split(TRAIN_ROOT, PACKED_ROOT / "TRAIN")
    pack_split(VAL_ROOT,   PACKED_ROOT / "VAL")
    pack_split(TEST_ROOT,  PACKED_ROOT / "TEST")

    print("\n[2/4] Loading data...")
    train_ds = PackedEEGDataset(PACKED_ROOT / "TRAIN")
    val_ds   = PackedEEGDataset(PACKED_ROOT / "VAL")

    train_loader = DataLoader(
        train_ds,
        batch_sampler=RunSampler(train_ds, BATCH_SIZE, shuffle=True, drop_last=False),
        collate_fn=collate_fn
    )

    val_loader = DataLoader(
        val_ds,
        batch_sampler=RunSampler(val_ds, BATCH_SIZE, shuffle=False, drop_last=False),
        collate_fn=collate_fn
    )

    sample_file = list((PACKED_ROOT / "TRAIN").glob("*.npz"))[0]
    sample = np.load(sample_file, allow_pickle=True)
    in_ch = int(sample["X_EEG"].shape[1])

    print("Input EEG channels:", in_ch)

    model = EEGModel(in_ch=in_ch, emb_dim=128, feat_dim=256, num_classes=NUM_CLASSES).to(DEVICE)

    # sanity check
    xb, yb, _, _ = next(iter(train_loader))
    xb = {k: v.to(DEVICE) for k, v in xb.items()}
    with torch.no_grad():
        logits, feat = model(xb, return_feat=True)
        print("Sanity logits shape:", logits.shape)
        print("Sanity feature shape:", feat.shape)  # expected: (B,256)

    print("\n[3/4] Training...")
    train(model, train_loader, val_loader)

    model.load_state_dict(torch.load(BEST_CKPT_PATH, map_location=DEVICE))

    print("\n[4/4] Extracting features...")
    extract(model, PACKED_ROOT / "TRAIN", FEAT_ROOT / "TRAIN")
    extract(model, PACKED_ROOT / "VAL",   FEAT_ROOT / "VAL")
    extract(model, PACKED_ROOT / "TEST",  FEAT_ROOT / "TEST")

    print("\n🎉 DONE")
    print("Features root:", FEAT_ROOT)

# =========================================================
if __name__ == "__main__":
    main()

CUDA available: True
GPU: NVIDIA GeForce RTX 5070
VRAM (GB): 11.94
Torch: 2.11.0+cu128

[1/4] Packing...


Packing EEG: 100%|██████████| 2316/2316 [09:32<00:00,  4.05it/s]


✅ Packed EEG: saved=2316, skipped=0


Packing EEG: 100%|██████████| 190/190 [00:49<00:00,  3.83it/s]


✅ Packed EEG: saved=190, skipped=0


...

sub-124_ses-01_run-82_win4s_pre900s_EEG.npz -> F shape = (736, 256)


Extract TEST: 100%|█████████▉| 343/344 [01:41<00:00, 14.10it/s]

sub-124_ses-01_run-83_win4s_pre900s_EEG.npz -> F shape = (741, 256)


Extract TEST: 100%|██████████| 344/344 [01:41<00:00,  3.40it/s]


sub-124_ses-01_run-84_win4s_pre900s_EEG.npz -> F shape = (747, 256)
✅ Features saved -> C:\SeniorProject\Data\NPZ_SPLITS_FILTERED_CAP1H_IMPD3\EEG_CNN_FEATURES\TEST

🎉 DONE
Features root: C:\SeniorProject\Data\NPZ_SPLITS_FILTERED_CAP1H_IMPD3\EEG_CNN_FEATURES


### Train BILSTM

In [1]:
from pathlib import Path
import json
import gc
import random
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

from sklearn.metrics import (
    f1_score,
    confusion_matrix,
    classification_report,
    balanced_accuracy_score,
    accuracy_score,
)

# =========================================================
# 0) Reproducibility
# =========================================================
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# =========================================================
# 1) Paths
# =========================================================

FEATURE_ROOT = Path(r"C:\SeniorProject\Data\NPZ_SPLITS_FILTERED_CAP1H_IMPD3\EEG_CNN_FEATURES")
TRAIN_ROOT = FEATURE_ROOT / "TRAIN"
VAL_ROOT   = FEATURE_ROOT / "VAL"
TEST_ROOT  = FEATURE_ROOT / "TEST"

OUT_ROOT = FEATURE_ROOT.parent / "EEG_BILSTM_RESULTS2"
OUT_ROOT.mkdir(parents=True, exist_ok=True)

BEST_CKPT_PATH = OUT_ROOT / "EEG_best_model.pt"
LAST_CKPT_PATH = OUT_ROOT / "EEG_last_model.pt"
METRICS_JSON   = OUT_ROOT / "EEG_final_metrics.json"
CLS_REPORT_TXT = OUT_ROOT / "EEG_test_classification_report.txt"
CM_NPY         = OUT_ROOT / "EEG_test_confusion_matrix.npy"
PRED_NPZ       = OUT_ROOT / "EEG_test_predictions.npz"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(torch.cuda.current_device())
    print("GPU:", props.name)
    print("VRAM (GB):", round(props.total_memory / (1024**3), 2))
print("Torch:", torch.__version__)

# =========================================================
# 2) Config
# =========================================================
NUM_CLASSES = 3
CLASS_NAMES = {0: "Normal", 1: "Preictal", 2: "Ictal"}

# ---------------------------------------------------------
# Sequence setup
# ---------------------------------------------------------
SEQ_LEN = 8
SEQ_STRIDE = 1
# ---------------------------------------------------------
# Training
# ---------------------------------------------------------
BATCH_SIZE = 96
MAX_EPOCHS = 90
MIN_EPOCHS = 40
PATIENCE = 10

LR = 7e-4
CLIP_NORM = 1.0

# fixed step schedule
LR_MILESTONES = [30, 60]
LR_GAMMA = 0.1

# ---------------------------------------------------------
# Model
# ---------------------------------------------------------
INPUT_DIM = 256
HIDDEN_SIZE = 192
NUM_LAYERS = 2
DROPOUT = 0.3
INPUT_DROPOUT = 0.15
# ---------------------------------------------------------
# LDAM-DRW
# ---------------------------------------------------------
LDAM_C = 0.5
DRW_START_EPOCH = 21   # صار أبكر بدل 61

NUM_WORKERS = 0
PIN_MEMORY = torch.cuda.is_available()

# =========================================================
# 3) Helpers
# =========================================================
def load_feature_run(path: Path):
    npz = np.load(path, allow_pickle=True)
    F_arr = np.asarray(npz["F"], dtype=np.float32)
    y_arr = np.asarray(npz["y"], dtype=np.int64)
    win_idx = np.asarray(npz["win_idx"], dtype=np.int32)
    base = str(npz["base"][0]) if "base" in npz else path.stem
    return F_arr, y_arr, win_idx, base

def compute_train_feature_stats(train_root: Path):
    sum_vec = None
    sumsq_vec = None
    total_n = 0

    files = sorted(train_root.glob("*.npz"))
    if len(files) == 0:
        raise RuntimeError(f"No feature files found in {train_root}")

    for p in files:
        F_arr, _, _, _ = load_feature_run(p)
        if F_arr.ndim != 2 or F_arr.shape[0] == 0:
            continue

        if sum_vec is None:
            sum_vec = F_arr.sum(axis=0, dtype=np.float64)
            sumsq_vec = (F_arr ** 2).sum(axis=0, dtype=np.float64)
        else:
            sum_vec += F_arr.sum(axis=0, dtype=np.float64)
            sumsq_vec += (F_arr ** 2).sum(axis=0, dtype=np.float64)

        total_n += F_arr.shape[0]

    if total_n == 0:
        raise RuntimeError("No valid TRAIN features found.")

    mean = sum_vec / total_n
    var = (sumsq_vec / total_n) - (mean ** 2)
    var = np.maximum(var, 1e-8)
    std = np.sqrt(var)

    return mean.astype(np.float32), std.astype(np.float32)

def compute_sequence_counts(root: Path, seq_len: int, seq_stride: int):
    counts = np.zeros(NUM_CLASSES, dtype=np.int64)

    for p in sorted(root.glob("*.npz")):
        _, y_arr, _, _ = load_feature_run(p)
        n = len(y_arr)
        if n < seq_len:
            continue

        for s in range(0, n - seq_len + 1, seq_stride):
            label = int(y_arr[s + seq_len - 1])
            counts[label] += 1

    counts = np.maximum(counts, 1)
    return counts

def make_paper_drw_weights(class_counts: np.ndarray) -> np.ndarray:
    """
    Inverse frequency, then normalized for stable scale.
    """
    counts = class_counts.astype(np.float64)
    weights = 1.0 / np.maximum(counts, 1.0)
    weights = weights / weights.mean()
    return weights.astype(np.float32)

def make_sequence_sample_weights(dataset, class_counts: np.ndarray) -> np.ndarray:
    """
    Weight per sequence sample for WeightedRandomSampler.
    Each sequence gets weight based on its sequence label.
    """
    class_weights = 1.0 / np.maximum(class_counts.astype(np.float64), 1.0)
    class_weights = class_weights / class_weights.sum()

    sample_weights = np.zeros(len(dataset.samples), dtype=np.float64)

    for i, (p, s) in enumerate(dataset.samples):
        _, y_arr, _, _ = dataset.cache[p]
        label = int(y_arr[s + dataset.seq_len - 1])
        sample_weights[i] = class_weights[label]

    return sample_weights.astype(np.float64)

# =========================================================
# 4) Dataset
# =========================================================
class SequenceFeatureDataset(Dataset):
    def __init__(self, root: Path, seq_len: int, seq_stride: int,
                 feat_mean: np.ndarray, feat_std: np.ndarray):
        self.root = Path(root)
        self.seq_len = int(seq_len)
        self.seq_stride = int(seq_stride)
        self.feat_mean = feat_mean.astype(np.float32)
        self.feat_std = feat_std.astype(np.float32)

        self.run_files = sorted(self.root.glob("*.npz"))
        if len(self.run_files) == 0:
            raise RuntimeError(f"No feature files found in {self.root}")

        self.samples = []
        self.cache = {}

        for p in self.run_files:
            F_arr, y_arr, win_idx, base = load_feature_run(p)

            if F_arr.ndim != 2:
                continue
            if len(y_arr) != F_arr.shape[0] or len(win_idx) != F_arr.shape[0]:
                continue

            n = F_arr.shape[0]
            if n < self.seq_len:
                continue

            self.cache[str(p)] = (F_arr, y_arr, win_idx, base)

            for s in range(0, n - self.seq_len + 1, self.seq_stride):
                self.samples.append((str(p), s))

        if len(self.samples) == 0:
            raise RuntimeError(f"No valid sequences found in {self.root}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        p, s = self.samples[idx]
        F_arr, y_arr, win_idx, base = self.cache[p]

        e = s + self.seq_len
        X_seq = F_arr[s:e]
        X_seq = (X_seq - self.feat_mean) / self.feat_std
        X_seq = np.nan_to_num(X_seq, nan=0.0, posinf=0.0, neginf=0.0)

        label = int(y_arr[e - 1])
        end_wi = int(win_idx[e - 1])

        return (
            torch.from_numpy(X_seq.astype(np.float32)),
            torch.tensor(label, dtype=torch.long),
            base,
            torch.tensor(end_wi, dtype=torch.long),
        )

def collate_seq(batch):
    Xs, ys, bases, end_wis = [], [], [], []
    for X, y, base, wi in batch:
        Xs.append(X)
        ys.append(y)
        bases.append(base)
        end_wis.append(wi)

    return (
        torch.stack(Xs, dim=0),
        torch.stack(ys, dim=0),
        bases,
        torch.stack(end_wis, dim=0)
    )

# =========================================================
# 5) LDAM Loss
# =========================================================
class LDAMLoss(nn.Module):
    """
    Delta_j = C / n_j^(1/4)
    """
    def __init__(self, class_counts, C=0.5, weight=None, s=30):
        super().__init__()

        counts = np.asarray(class_counts, dtype=np.float32)
        counts = np.maximum(counts, 1.0)

        m_list = C / np.power(counts, 0.25)
        m_list = torch.tensor(m_list, dtype=torch.float32)

        self.register_buffer("m_list", m_list)
        self.s = float(s)

        if weight is not None:
            weight = torch.as_tensor(weight, dtype=torch.float32)
            self.register_buffer("weight", weight)
        else:
            self.weight = None

    def set_weight(self, weight):
        if weight is None:
            self.weight = None
        else:
            weight = torch.as_tensor(weight, dtype=torch.float32, device=self.m_list.device)
            self.weight = weight

    def forward(self, logits, target):
        target = target.long()
        index = torch.zeros_like(logits, dtype=torch.bool)
        index.scatter_(1, target.view(-1, 1), True)

        batch_m = self.m_list[target].view(-1, 1)
        logits_m = logits - batch_m * index.float()

        return F.cross_entropy(self.s * logits_m, target, weight=self.weight)

# =========================================================
# 6) Model
# =========================================================
class FinalBiLSTM(nn.Module):
    def __init__(self, input_dim=256, hidden_size=256, num_layers=2, dropout=0.5, num_classes=3, input_dropout=0.2):
        super().__init__()

        self.input_norm = nn.LayerNorm(input_dim)
        self.input_dropout = nn.Dropout(input_dropout)

        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if num_layers > 1 else 0.0
        )

        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_size * 2, num_classes)

    def forward(self, x):
        x = self.input_norm(x)
        x = self.input_dropout(x)
        h, _ = self.lstm(x)
        last_h = h[:, -1, :]
        last_h = self.dropout(last_h)
        logits = self.classifier(last_h)
        return logits

# =========================================================
# 7) Train / Eval
# =========================================================
@torch.no_grad()
def evaluate(model, loader, device, loss_fn):
    model.eval()

    total_loss = 0.0
    n = 0
    all_y = []
    all_p = []
    all_probs = []
    all_bases = []
    all_end_wis = []

    for X, y, bases, end_wis in loader:
        X = X.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        logits = model(X)
        loss = loss_fn(logits, y)

        if not torch.isfinite(loss):
            raise RuntimeError("Validation/Test loss became NaN or Inf.")

        probs = torch.softmax(logits, dim=1)
        pred = torch.argmax(probs, dim=1)

        total_loss += float(loss.item()) * y.size(0)
        n += y.size(0)

        all_y.append(y.detach().cpu().numpy())
        all_p.append(pred.detach().cpu().numpy())
        all_probs.append(probs.detach().cpu().numpy())
        all_bases.extend(bases)
        all_end_wis.append(end_wis.detach().cpu().numpy())

    y_true = np.concatenate(all_y, axis=0)
    y_pred = np.concatenate(all_p, axis=0)
    probs = np.concatenate(all_probs, axis=0)
    end_wis = np.concatenate(all_end_wis, axis=0)

    avg_loss = total_loss / max(1, n)

    return {
        "loss": avg_loss,
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "weighted_f1": f1_score(y_true, y_pred, average="weighted", zero_division=0),
        "balanced_acc": balanced_accuracy_score(y_true, y_pred),
        "y_true": y_true,
        "y_pred": y_pred,
        "probs": probs,
        "bases": np.asarray(all_bases, dtype=object),
        "end_wis": end_wis
    }

def train_one_epoch(model, loader, optimizer, device, loss_fn, clip_norm=1.0):
    model.train()

    total_loss = 0.0
    n = 0

    for X, y, _, _ in loader:
        X = X.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        logits = model(X)
        loss = loss_fn(logits, y)

        if not torch.isfinite(loss):
            raise RuntimeError("Training loss became NaN or Inf.")

        loss.backward()

        if clip_norm is not None:
            torch.nn.utils.clip_grad_norm_(model.parameters(), clip_norm)

        optimizer.step()

        total_loss += float(loss.item()) * y.size(0)
        n += y.size(0)

    return total_loss / max(1, n)

# =========================================================
# 8) Main
# =========================================================
def main():
    feat_mean, feat_std = compute_train_feature_stats(TRAIN_ROOT)
    print("Feature dim:", len(feat_mean))

    train_ds = SequenceFeatureDataset(TRAIN_ROOT, SEQ_LEN, SEQ_STRIDE, feat_mean, feat_std)
    val_ds   = SequenceFeatureDataset(VAL_ROOT,   SEQ_LEN, SEQ_STRIDE, feat_mean, feat_std)
    test_ds  = SequenceFeatureDataset(TEST_ROOT,  SEQ_LEN, SEQ_STRIDE, feat_mean, feat_std)

    class_counts = compute_sequence_counts(TRAIN_ROOT, SEQ_LEN, SEQ_STRIDE)
    print("TRAIN sequence label counts:", {i: int(class_counts[i]) for i in range(NUM_CLASSES)})

    drw_weights = make_paper_drw_weights(class_counts)
    print("Normalized DRW class weights:", drw_weights.tolist())

    # -----------------------------------------------------
    # WeightedRandomSampler
    # -----------------------------------------------------
    train_sample_weights = make_sequence_sample_weights(train_ds, class_counts)
    train_sampler = WeightedRandomSampler(
        weights=torch.as_tensor(train_sample_weights, dtype=torch.double),
        num_samples=len(train_sample_weights),
        replacement=True
    )

    train_loader = DataLoader(
        train_ds,
        batch_size=BATCH_SIZE,
        sampler=train_sampler,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        drop_last=False,
        collate_fn=collate_seq
    )
    val_loader = DataLoader(
        val_ds,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        drop_last=False,
        collate_fn=collate_seq
    )
    test_loader = DataLoader(
        test_ds,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        drop_last=False,
        collate_fn=collate_seq
    )

    print("Train sequences:", len(train_ds))
    print("Val sequences  :", len(val_ds))
    print("Test sequences :", len(test_ds))
    print("Sampler enabled: WeightedRandomSampler")

    loss_fn = LDAMLoss(
        class_counts=class_counts,
        C=LDAM_C,
        weight=None,
        s=30
    ).to(DEVICE)

    model = FinalBiLSTM(
        input_dim=INPUT_DIM,
        hidden_size=HIDDEN_SIZE,
        num_layers=NUM_LAYERS,
        dropout=DROPOUT,
        num_classes=NUM_CLASSES,
        input_dropout=INPUT_DROPOUT
    ).to(DEVICE)

    optimizer = torch.optim.Adam(model.parameters(), lr=LR)

    scheduler = torch.optim.lr_scheduler.MultiStepLR(
        optimizer,
        milestones=LR_MILESTONES,
        gamma=LR_GAMMA
    )

    X0, _, _, _ = next(iter(train_loader))
    X0 = X0.to(DEVICE)
    with torch.no_grad():
        logits0 = model(X0)
    print("Sanity logits min/max:", logits0.min().item(), logits0.max().item())
    print("Sanity logits finite :", torch.isfinite(logits0).all().item())

    best_macro_f1 = -1.0
    best_bal_acc = -1.0
    best_val_loss = float("inf")
    best_epoch = -1
    bad_epochs = 0

    MACRO_EPS = 1e-3
    BAL_EPS = 1e-4
    LOSS_EPS = 1e-4

    print("\n[1/2] Training LDAM-DRW + WeightedRandomSampler...")
    for epoch in range(1, MAX_EPOCHS + 1):
        if epoch == DRW_START_EPOCH:
            print(f"--> DRW activated at epoch {epoch}")
            loss_fn.set_weight(drw_weights)

        current_weight_status = "ON" if loss_fn.weight is not None else "OFF"

        train_loss = train_one_epoch(
            model=model,
            loader=train_loader,
            optimizer=optimizer,
            device=DEVICE,
            loss_fn=loss_fn,
            clip_norm=CLIP_NORM
        )

        val_out = evaluate(
            model=model,
            loader=val_loader,
            device=DEVICE,
            loss_fn=loss_fn
        )

        val_loss = val_out["loss"]
        val_macro = val_out["macro_f1"]
        val_bal_acc = val_out["balanced_acc"]
        val_weighted = val_out["weighted_f1"]

        lr_now = optimizer.param_groups[0]["lr"]

        print(
            f"Epoch {epoch:03d} | "
            f"DRW {current_weight_status} | "
            f"lr {lr_now:.2e} | "
            f"train_loss {train_loss:.4f} | "
            f"val_loss {val_loss:.4f} | "
            f"val_bal_acc {val_bal_acc:.4f} | "
            f"val_macro_f1 {val_macro:.4f} | "
            f"val_weighted_f1 {val_weighted:.4f}"
        )

        torch.save({
            "epoch": epoch,
            "model": model.state_dict(),
            "optim": optimizer.state_dict(),
        }, LAST_CKPT_PATH)

        improved = False

        if val_macro > best_macro_f1 + MACRO_EPS:
            improved = True
        elif abs(val_macro - best_macro_f1) <= MACRO_EPS:
            if val_bal_acc > best_bal_acc + BAL_EPS:
                improved = True
            elif abs(val_bal_acc - best_bal_acc) <= BAL_EPS:
                if val_loss < best_val_loss - LOSS_EPS:
                    improved = True

        if improved:
            best_macro_f1 = val_macro
            best_bal_acc = val_bal_acc
            best_val_loss = val_loss
            best_epoch = epoch
            bad_epochs = 0

            torch.save({
                "epoch": epoch,
                "model": model.state_dict(),
                "optim": optimizer.state_dict(),
                "best_macro_f1": best_macro_f1,
                "best_bal_acc": best_bal_acc,
                "best_val_loss": best_val_loss,
                "seq_len": SEQ_LEN,
                "input_dim": INPUT_DIM,
                "hidden_size": HIDDEN_SIZE,
                "num_layers": NUM_LAYERS,
                "dropout": DROPOUT,
                "input_dropout": INPUT_DROPOUT,
                "lr": LR,
                "ldam_C": LDAM_C,
                "drw_start_epoch": DRW_START_EPOCH,
                "lr_milestones": LR_MILESTONES,
                "lr_gamma": LR_GAMMA
            }, BEST_CKPT_PATH)

            print(
                f"✅ Saved BEST checkpoint "
                f"(macro_f1={best_macro_f1:.4f}, bal_acc={best_bal_acc:.4f}, val_loss={best_val_loss:.4f})"
            )
        else:
            bad_epochs += 1

        early_stop_ready = epoch >= max(MIN_EPOCHS, DRW_START_EPOCH + 10)

        if early_stop_ready and bad_epochs >= PATIENCE:
            print(f"🛑 Early stopping at epoch {epoch}. Best epoch={best_epoch}")
            break

        scheduler.step()

        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    print("\n[2/2] Final TEST evaluation...")
    ckpt = torch.load(BEST_CKPT_PATH, map_location=DEVICE)
    model.load_state_dict(ckpt["model"])

    if ckpt["epoch"] >= DRW_START_EPOCH:
        loss_fn.set_weight(drw_weights)
    else:
        loss_fn.set_weight(None)

    test_out = evaluate(model, test_loader, DEVICE, loss_fn)

    y_true = test_out["y_true"]
    y_pred = test_out["y_pred"]
    probs  = test_out["probs"]
    bases  = test_out["bases"]
    end_wis = test_out["end_wis"]

    test_loss = test_out["loss"]
    acc = accuracy_score(y_true, y_pred)
    bal_acc = test_out["balanced_acc"]
    macro_f1 = test_out["macro_f1"]
    weighted_f1 = test_out["weighted_f1"]

    cm = confusion_matrix(y_true, y_pred)
    report = classification_report(y_true, y_pred, digits=4, zero_division=0)

    print("\n===== FINAL TEST RESULTS =====")
    print("Best epoch             :", ckpt["epoch"])
    print("Test loss              :", test_loss)
    print("Test accuracy          :", acc)
    print("Test balanced accuracy :", bal_acc)
    print("Test macro_f1          :", macro_f1)
    print("Test weighted_f1       :", weighted_f1)
    print("Confusion matrix:\n", cm)
    print("\nClassification report:\n", report)

    with open(CLS_REPORT_TXT, "w", encoding="utf-8") as f:
        f.write(report)

    np.save(CM_NPY, cm)

    np.savez_compressed(
        PRED_NPZ,
        y_true=y_true.astype(np.int64),
        y_pred=y_pred.astype(np.int64),
        probs=probs.astype(np.float32),
        bases=bases,
        end_wis=end_wis.astype(np.int64),
    )

    metrics = {
        "best_epoch": int(ckpt["epoch"]),
        "test_loss": float(test_loss),
        "test_accuracy": float(acc),
        "test_balanced_accuracy": float(bal_acc),
        "test_macro_f1": float(macro_f1),
        "test_weighted_f1": float(weighted_f1),
        "best_val_macro_f1": float(ckpt.get("best_macro_f1", -1.0)),
        "best_val_bal_acc": float(ckpt.get("best_bal_acc", -1.0)),
        "best_val_loss": float(ckpt.get("best_val_loss", -1.0)),
        "seq_len": int(SEQ_LEN),
        "input_dim": int(INPUT_DIM),
        "hidden_size": int(HIDDEN_SIZE),
        "num_layers": int(NUM_LAYERS),
        "dropout": float(DROPOUT),
        "input_dropout": float(INPUT_DROPOUT),
        "lr": float(LR),
        "ldam_C": float(LDAM_C),
        "drw_start_epoch": int(DRW_START_EPOCH),
        "lr_milestones": LR_MILESTONES,
        "lr_gamma": float(LR_GAMMA),
    }

    with open(METRICS_JSON, "w", encoding="utf-8") as f:
        json.dump(metrics, f, indent=2)

    print("\n✅ All done.")
    print("Results folder:", OUT_ROOT)

if __name__ == "__main__":
    main()

CUDA available: True
GPU: NVIDIA GeForce RTX 5070
VRAM (GB): 11.94
Torch: 2.11.0+cu128
Feature dim: 256
TRAIN sequence label counts: {0: 1704691, 1: 127548, 2: 11053}
Normalized DRW class weights: [0.017794238403439522, 0.23782166838645935, 2.7443840503692627]
Train sequences: 1843292
Val sequences  : 154213
Test sequences : 1410281
Sampler enabled: WeightedRandomSampler
Sanity logits min/max: -0.13219276070594788 0.1730070561170578
Sanity logits finite : True

[1/2] Training LDAM-DRW + WeightedRandomSampler...
Epoch 001 | DRW OFF | lr 7.00e-04 | train_loss 0.6085 | val_loss 1.8439 | val_bal_acc 0.3243 | val_macro_f1 0.2539 | val_weighted_f1 0.5896
✅ Saved BEST checkpoint (macro_f1=0.2539, bal_acc=0.3243, val_loss=1.8439)
Epoch 002 | DRW OFF | lr 7.00e-04 | train_loss 0.3734 | val_loss 1.9895 | val_bal_acc 0.3481 | val_macro_f1 0.2787 | val_weighted_f1 0.6443
✅ Saved BEST checkpoint (macro_f1=0.2787, bal_acc=0.3481, val_loss=1.9895)
Epoch 003 | DRW OFF | lr 7.00e-04 | train_loss 0.3062

### Result

In [2]:
from pathlib import Path
import json
import numpy as np
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    precision_score,
    recall_score
)

# =========================================================
# 1) Paths
# =========================================================
OUT_ROOT = Path(r"C:\SeniorProject\Data\NPZ_SPLITS_FILTERED_CAP1H_IMPD3\EEG_BILSTM_RESULTS2")

METRICS_JSON = OUT_ROOT / "EEG_final_metrics.json"
PRED_NPZ     = OUT_ROOT / "EEG_test_predictions.npz"
CM_NPY       = OUT_ROOT / "EEG_test_confusion_matrix.npy"

CLASS_NAMES = {
    0: "Normal",
    1: "Preictal",
    2: "Ictal"
}

# =========================================================
# 2) Load saved results
# =========================================================
with open(METRICS_JSON, "r", encoding="utf-8") as f:
    saved_metrics = json.load(f)

pred_data = np.load(PRED_NPZ, allow_pickle=True)
y_true = pred_data["y_true"]
y_pred = pred_data["y_pred"]
probs  = pred_data["probs"]
bases  = pred_data["bases"]
end_wis = pred_data["end_wis"]

cm = np.load(CM_NPY)

# =========================================================
# 3) General metrics
# =========================================================
print("=" * 70)
print("GENERAL METRICS")
print("=" * 70)

print(f"Best epoch             : {saved_metrics.get('best_epoch')}")
print(f"Test loss              : {saved_metrics.get('test_loss'):.6f}")
print(f"Test accuracy          : {saved_metrics.get('test_accuracy'):.6f}")
print(f"Test balanced accuracy : {saved_metrics.get('test_balanced_accuracy'):.6f}")
print(f"Test macro F1          : {saved_metrics.get('test_macro_f1'):.6f}")
print(f"Test weighted F1       : {saved_metrics.get('test_weighted_f1'):.6f}")

print("\nExtra recomputed metrics:")
print(f"Accuracy               : {accuracy_score(y_true, y_pred):.6f}")
print(f"Balanced accuracy      : {balanced_accuracy_score(y_true, y_pred):.6f}")
print(f"Macro precision        : {precision_score(y_true, y_pred, average='macro', zero_division=0):.6f}")
print(f"Macro recall           : {recall_score(y_true, y_pred, average='macro', zero_division=0):.6f}")
print(f"Macro F1               : {f1_score(y_true, y_pred, average='macro', zero_division=0):.6f}")
print(f"Weighted precision     : {precision_score(y_true, y_pred, average='weighted', zero_division=0):.6f}")
print(f"Weighted recall        : {recall_score(y_true, y_pred, average='weighted', zero_division=0):.6f}")
print(f"Weighted F1            : {f1_score(y_true, y_pred, average='weighted', zero_division=0):.6f}")

# =========================================================
# 4) Confusion matrix
# =========================================================
print("\n" + "=" * 70)
print("CONFUSION MATRIX")
print("=" * 70)
print("Rows = True labels, Columns = Predicted labels\n")

header = " " * 12 + "".join([f"{CLASS_NAMES[i]:>12}" for i in range(len(CLASS_NAMES))])
print(header)

for i in range(len(CLASS_NAMES)):
    row_str = f"{CLASS_NAMES[i]:>12}" + "".join([f"{cm[i, j]:>12d}" for j in range(len(CLASS_NAMES))])
    print(row_str)

# =========================================================
# 5) Classification report
# =========================================================
print("\n" + "=" * 70)
print("SKLEARN CLASSIFICATION REPORT")
print("=" * 70)

report = classification_report(
    y_true,
    y_pred,
    target_names=[CLASS_NAMES[i] for i in range(len(CLASS_NAMES))],
    digits=6,
    zero_division=0
)
print(report)

# =========================================================
# 6) Per-class detailed metrics: TP, FP, FN, TN, specificity...
# =========================================================
print("\n" + "=" * 70)
print("PER-CLASS DETAILED METRICS")
print("=" * 70)

num_classes = len(CLASS_NAMES)
total_samples = len(y_true)

for c in range(num_classes):
    tp = cm[c, c]
    fp = cm[:, c].sum() - tp
    fn = cm[c, :].sum() - tp
    tn = cm.sum() - (tp + fp + fn)

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0   # sensitivity
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    npv = tn / (tn + fn) if (tn + fn) > 0 else 0.0
    f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0.0
    class_acc = (tp + tn) / total_samples if total_samples > 0 else 0.0

    support_true = (y_true == c).sum()
    support_pred = (y_pred == c).sum()

    print(f"\nClass {c} -> {CLASS_NAMES[c]}")
    print("-" * 50)
    print(f"Support (true)         : {support_true}")
    print(f"Predicted as class     : {support_pred}")
    print(f"TP                     : {tp}")
    print(f"FP                     : {fp}")
    print(f"FN                     : {fn}")
    print(f"TN                     : {tn}")
    print(f"Precision              : {precision:.6f}")
    print(f"Recall / Sensitivity   : {recall:.6f}")
    print(f"Specificity            : {specificity:.6f}")
    print(f"NPV                    : {npv:.6f}")
    print(f"F1-score               : {f1:.6f}")
    print(f"One-vs-Rest Accuracy   : {class_acc:.6f}")

# =========================================================
# 7) Distribution of true and predicted labels
# =========================================================
print("\n" + "=" * 70)
print("LABEL DISTRIBUTIONS")
print("=" * 70)

for c in range(num_classes):
    n_true = int((y_true == c).sum())
    n_pred = int((y_pred == c).sum())
    print(f"{CLASS_NAMES[c]:<10} | true: {n_true:<8} | pred: {n_pred:<8}")

# =========================================================
# 8) Show some prediction examples
# =========================================================
print("\n" + "=" * 70)
print("FIRST 20 PREDICTIONS")
print("=" * 70)

n_show = min(20, len(y_true))
for i in range(n_show):
    true_name = CLASS_NAMES[int(y_true[i])]
    pred_name = CLASS_NAMES[int(y_pred[i])]
    prob_str = ", ".join([f"{CLASS_NAMES[j]}={probs[i][j]:.4f}" for j in range(num_classes)])
    print(
        f"[{i:03d}] base={bases[i]} | end_wi={int(end_wis[i])} | "
        f"true={true_name} | pred={pred_name} | probs: {prob_str}"
    )

GENERAL METRICS
Best epoch             : 13
Test loss              : 1.197641
Test accuracy          : 0.726504
Test balanced accuracy : 0.423937
Test macro F1          : 0.302149
Test weighted F1       : 0.828342

Extra recomputed metrics:
Accuracy               : 0.726504
Balanced accuracy      : 0.423937
Macro precision        : 0.340366
Macro recall           : 0.423937
Macro F1               : 0.302149
Weighted precision     : 0.970911
Weighted recall        : 0.726504
Weighted F1            : 0.828342

CONFUSION MATRIX
Rows = True labels, Columns = Predicted labels

                  Normal    Preictal       Ictal
      Normal     1017672      349550       20567
    Preictal       13258        6542        1090
       Ictal         824         417         361

SKLEARN CLASSIFICATION REPORT
              precision    recall  f1-score   support

      Normal   0.986351  0.733305  0.841210   1387789
    Preictal   0.018350  0.313164  0.034669     20890
       Ictal   0.016396  0.2253

In [ ]:
from pathlib import Path
import numpy as np

OUT_ROOT = Path(r"C:\SeniorProject\Data\NPZ_SPLITS_FILTERED_CAP1H_IMPD3\EEG_BILSTM_RESULTS2")

cm = np.load(OUT_ROOT / "EEG_test_confusion_matrix.npy")

CLASS_NAMES = {
    0: "Normal",
    1: "Preictal",
    2: "Ictal"
}

print("=" * 80)
print(f"{'Class':<12} {'TP':>8} {'FP':>8} {'FN':>8} {'TN':>8} {'TPR':>10} {'FPR':>10} {'FNR':>10} {'TNR':>10} {'Prec':>10}")
print("=" * 80)

for c in range(len(CLASS_NAMES)):
    tp = cm[c, c]
    fp = cm[:, c].sum() - tp
    fn = cm[c, :].sum() - tp
    tn = cm.sum() - tp - fp - fn

    tpr = tp / (tp + fn) if (tp + fn) > 0 else 0.0   # Recall / Sensitivity
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0   # False Positive Rate
    fnr = fn / (fn + tp) if (fn + tp) > 0 else 0.0   # False Negative Rate
    tnr = tn / (tn + fp) if (tn + fp) > 0 else 0.0   # Specificity
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0  # Precision

    print(f"{CLASS_NAMES[c]:<12} {tp:>8d} {fp:>8d} {fn:>8d} {tn:>8d} {tpr:>10.4f} {fpr:>10.4f} {fnr:>10.4f} {tnr:>10.4f} {prec:>10.4f}")

Class              TP       FP       FN       TN        TPR        FPR        FNR        TNR       Prec
Normal        1017672    14082   370117     8410     0.7333     0.6261     0.2667     0.3739     0.9864
Preictal         6542   349967    14348  1039424     0.3132     0.2519     0.6868     0.7481     0.0184
Ictal             361    21657     1241  1387022     0.2253     0.0154     0.7747     0.9846     0.0164
